In [ ]:
"""
Ghost Car — Zona de Màxim Gap (auto-detectada)
Troba automàticament la finestra del circuit on s'acumula més diferència de temps
i genera la visualització detallada de traçat + telemetria.
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.collections import LineCollection
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT   = r"c:\Users\AvAd738\OneDrive - HP Inc\Documents\Adriana\AI-coaching-for-iRacing\notebooks"
DATA   = os.path.join(ROOT, "data", "test dani")
F_SLOW = os.path.join(DATA,
    "Garage 61 - Dani Gracia - Ferrari 296 GT3 - Autodromo Internazionale "
    "Enzo e Dino Ferrari (Grand Prix) - 01.44.845 - 01KQ0DW522RXG6H7Y5K7JH28F5.csv")
F_FAST = os.path.join(DATA,
    "Garage 61 - Dani Gracia - Ferrari 296 GT3 - Autodromo Internazionale "
    "Enzo e Dino Ferrari (Grand Prix) - 01.42.797 - 01KQ0DW522RXG6H7Y5K9WPXCRR.csv")
OUT    = os.path.join(ROOT, "ghost_maxgap_visualization.png")

TRACK_LEN = 4909   # metres Imola GP

# ── Palette ───────────────────────────────────────────────────────────────────
BG    = '#07080F'
RED   = '#FF3B5C'
CYAN  = '#00E5C0'
GOLD  = '#FFD040'
DIM   = '#3A3A55'
TXT   = '#C8CCDF'
GREEN = '#39FF14'   # accent for "max gap" callouts

cm_slow = mcolors.LinearSegmentedColormap.from_list(
    'sl', ['#7a0020', RED, '#FF9060'])
cm_fast = mcolors.LinearSegmentedColormap.from_list(
    'fl', ['#005540', CYAN, '#80FFE8'])

# ── Load & preprocess ─────────────────────────────────────────────────────────
_raw_s = pd.read_csv(F_SLOW)
_raw_f = pd.read_csv(F_FAST)

LAT0    = (_raw_s['Lat'].iloc[0] + _raw_f['Lat'].iloc[0]) / 2
LON0    = (_raw_s['Lon'].iloc[0] + _raw_f['Lon'].iloc[0]) / 2
COS_LAT = np.cos(np.radians(LAT0))


def process(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if df['LapDistPct'].iloc[-1] < df['LapDistPct'].iloc[-2]:
        df = df.iloc[:-1].reset_index(drop=True)
    df['X'] = (df['Lon'] - LON0) * COS_LAT * 111_320
    df['Y'] = (df['Lat'] - LAT0) * 111_320
    df['X'] = gaussian_filter1d(df['X'].values, sigma=2)
    df['Y'] = gaussian_filter1d(df['Y'].values, sigma=2)
    dpct    = df['LapDistPct'].diff().clip(lower=0).fillna(0)
    vms     = df['Speed'].clip(lower=0.5)  # m/s
    df['T'] = (dpct * TRACK_LEN / vms).cumsum()
    return df


ds = process(_raw_s)
df = process(_raw_f)

# ── Dense interpolation grid ──────────────────────────────────────────────────
N_PTS = 8000
pct   = np.linspace(0.0, 1.0, N_PTS)


def interp_lap(df_in: pd.DataFrame) -> dict:
    d = df_in.drop_duplicates('LapDistPct').sort_values('LapDistPct')
    def fi(col):
        return interp1d(d['LapDistPct'], d[col], bounds_error=False,
                        fill_value=(float(d[col].iloc[0]),
                                    float(d[col].iloc[-1])))(pct)
    return {c: fi(c) for c in
            ['X', 'Y', 'Speed', 'T', 'Brake', 'Throttle', 'SteeringWheelAngle']}


S = interp_lap(ds)
F = interp_lap(df)

tgap       = S['T'] - F['T']
total_gap  = float(tgap.max())

# ── Auto-detect max-gap zone (sliding window 5% of lap) ───────────────────────
WIN_PCT = 0.05
win     = int(N_PTS * WIN_PCT)
growth  = np.array([tgap[i + win] - tgap[i] for i in range(N_PTS - win)])

best_i  = int(np.argmax(growth))
Z_START = float(pct[best_i])
Z_END   = float(pct[best_i + win])
Z_GAP   = float(growth[best_i])
Z_PCT_TOTAL = Z_GAP / total_gap * 100

# Context window (4% before, 4% after)
VIEW_START = max(0.0, Z_START - 0.045)
VIEW_END   = min(1.0, Z_END   + 0.045)

print("OK Zona maxim gap: %.1f%% - %.1f%%  -> +%.3fs  (%.0f%% del gap total)" % (Z_START*100, Z_END*100, Z_GAP, Z_PCT_TOTAL))

# ── Masks ─────────────────────────────────────────────────────────────────────
view_mask = (pct >= VIEW_START) & (pct <= VIEW_END)
zone_mask = (pct >= Z_START)    & (pct <= Z_END)

pct_v  = pct[view_mask]
sx_v   = S['X'][view_mask];    sy_v   = S['Y'][view_mask]
sspd_v = S['Speed'][view_mask] * 3.6
sbrk_v = S['Brake'][view_mask] * 100
sthr_v = S['Throttle'][view_mask] * 100
sstr_v = np.degrees(S['SteeringWheelAngle'][view_mask])
fx_v   = F['X'][view_mask];    fy_v   = F['Y'][view_mask]
fspd_v = F['Speed'][view_mask] * 3.6
fbrk_v = F['Brake'][view_mask] * 100
fthr_v = F['Throttle'][view_mask] * 100
fstr_v = np.degrees(F['SteeringWheelAngle'][view_mask])
tgap_v = tgap[view_mask]

# ── Ghost car positions (time-synchronized) ────────────────────────────────────
def mk_t2pct(t_arr):
    si  = np.argsort(t_arr)
    t_s = t_arr[si]; p_s = pct[si]
    _, u = np.unique(t_s, return_index=True)
    return interp1d(t_s[u], p_s[u],
                    bounds_error=False, fill_value=(p_s[0], p_s[-1]))


t2p_f = mk_t2pct(F['T'])
t2p_s = mk_t2pct(S['T'])
p2sx  = interp1d(pct, S['X'], bounds_error=False, fill_value='extrapolate')
p2sy  = interp1d(pct, S['Y'], bounds_error=False, fill_value='extrapolate')
p2fx  = interp1d(pct, F['X'], bounds_error=False, fill_value='extrapolate')
p2fy  = interp1d(pct, F['Y'], bounds_error=False, fill_value='extrapolate')

T_zs     = float(interp1d(pct, F['T'])(Z_START))
T_ze     = float(interp1d(pct, F['T'])(Z_END))
ghost_Ts = np.linspace(T_zs, T_ze, 10)

ghosts = []
for T in ghost_Ts:
    pf = float(t2p_f(T))
    ps = float(t2p_s(T))
    if VIEW_START <= max(pf, ps) <= VIEW_END:
        ghosts.append(dict(
            T=T, pf=pf, ps=ps,
            fx=float(p2fx(pf)), fy=float(p2fy(pf)),
            sx=float(p2sx(ps)), sy=float(p2sy(ps)),
            gap_m=(pf - ps) * TRACK_LEN,
            gap_t=float(np.interp(pf, pct, tgap)),
        ))

# ── Gap rate for the gap-rate mini chart ──────────────────────────────────────
dgap_raw    = np.gradient(tgap, pct) * (1 / TRACK_LEN)   # s/m
dgap_smooth = gaussian_filter1d(dgap_raw, sigma=30)

# ── Helpers ───────────────────────────────────────────────────────────────────
def seg_lc(x, y, spd, cmap, lw=3.2):
    pts  = np.array([x, y]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    norm = plt.Normalize(np.percentile(spd, 2), np.percentile(spd, 98))
    lc   = LineCollection(segs, cmap=cmap, norm=norm,
                          linewidth=lw, capstyle='round', alpha=0.97)
    lc.set_array(spd[:-1])
    return lc


def style_ax(a, show_zone=True):
    a.set_facecolor(BG)
    for sp in ['top', 'right']:
        a.spines[sp].set_visible(False)
    for sp in ['bottom', 'left']:
        a.spines[sp].set_color(DIM)
    a.tick_params(colors=DIM, labelcolor=TXT, labelsize=7)
    a.grid(axis='y', color='#181828', linewidth=0.5, alpha=0.8)
    a.set_xlim(VIEW_START, VIEW_END)
    if show_zone:
        a.axvspan(Z_START, Z_END, color=GREEN, alpha=0.07, zorder=0)
        a.axvline(Z_START, color=GREEN, lw=0.9, ls='--', alpha=0.55)
        a.axvline(Z_END,   color=GREEN, lw=0.9, ls='--', alpha=0.55)


# ── Figure ────────────────────────────────────────────────────────────────────
plt.rcParams.update({'font.family': 'monospace', 'font.size': 9})

fig = plt.figure(figsize=(24, 15), facecolor=BG, dpi=150)
gs  = GridSpec(5, 5, fig,
               left=0.03, right=0.975, top=0.91, bottom=0.055,
               hspace=0.55, wspace=0.30,
               width_ratios=[1.7, 1.7, 0.12, 1.3, 1.3],
               height_ratios=[1, 1, 1, 1, 0.75])

# ═══════════════════════════════════════════════════════════════════════════
#  CIRCUIT MAP ZOOM  (left 2 cols × top 4 rows)
# ═══════════════════════════════════════════════════════════════════════════
ax_map = fig.add_subplot(gs[:4, :2])
ax_map.set_facecolor(BG)
ax_map.set_aspect('equal', adjustable='datalim')

# ── Dim context road ─────────────────────────────────────────────────────
ax_map.plot(sx_v, sy_v, color='#3a1a25', lw=8, solid_capstyle='round', zorder=1, alpha=0.6)
ax_map.plot(fx_v, fy_v, color='#0a2a22', lw=8, solid_capstyle='round', zorder=1, alpha=0.6)

# ── Zone highlight road surface ───────────────────────────────────────────
z_sx = S['X'][zone_mask]; z_sy = S['Y'][zone_mask]
z_fx = F['X'][zone_mask]; z_fy = F['Y'][zone_mask]
ax_map.plot(z_sx, z_sy, color='#3d0018', lw=14, solid_capstyle='round', zorder=2)
ax_map.plot(z_fx, z_fy, color='#003322', lw=14, solid_capstyle='round', zorder=2)

# ── Speed-coloured traces ─────────────────────────────────────────────────
ax_map.add_collection(seg_lc(sx_v, sy_v, sspd_v, cm_slow, lw=4.0))
ax_map.add_collection(seg_lc(fx_v, fy_v, fspd_v, cm_fast, lw=2.8))

# ── Direction arrows ─────────────────────────────────────────────────────
for ap in [VIEW_START + 0.015, (VIEW_START + VIEW_END) / 2, VIEW_END - 0.015]:
    idx = np.argmin(np.abs(pct_v - ap))
    if idx + 6 < len(pct_v):
        dx = fx_v[idx + 6] - fx_v[idx]; dy = fy_v[idx + 6] - fy_v[idx]
        d  = np.hypot(dx, dy)
        if d > 0.5:
            ax_map.annotate('',
                xy=(fx_v[idx] + dx, fy_v[idx] + dy),
                xytext=(fx_v[idx], fy_v[idx]),
                arrowprops=dict(arrowstyle='->', color=CYAN,
                                lw=1.6, mutation_scale=14),
                zorder=10)

# ── Ghost markers ─────────────────────────────────────────────────────────
shown = []
for i, g in enumerate(ghosts):
    sx_, sy_, fx_, fy_ = g['sx'], g['sy'], g['fx'], g['fy']
    dist = np.hypot(fx_ - sx_, fy_ - sy_)

    # Declutter
    if any(np.hypot(sx_ - ox, sy_ - oy) < 45 for ox, oy in shown):
        continue
    shown.append((sx_, sy_))

    if dist > 0.8:
        ax_map.plot([sx_, fx_], [sy_, fy_],
                    color=GREEN, lw=1.8, ls='--', alpha=0.70, zorder=5)

    ax_map.scatter(sx_, sy_, s=140, color=RED,  edgecolors='white',
                   linewidths=0.9, zorder=7, marker='o')
    ax_map.scatter(fx_, fy_, s=140, color=CYAN, edgecolors='white',
                   linewidths=0.9, zorder=7, marker='D')

    if i % 2 == 0 and dist > 2.5:
        mid_x = (sx_ + fx_) / 2; mid_y = (sy_ + fy_) / 2
        ax_map.text(mid_x + 2, mid_y + 2,
                    '+%.0fm' % g['gap_m'],
                    color=GREEN, fontsize=6.5, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', fc='#080f0a',
                              ec=GREEN, alpha=0.88),
                    zorder=8)

# ── Max gap ghost highlight ────────────────────────────────────────────────
if ghosts:
    mg = max(ghosts, key=lambda g_: g_['gap_m'])
    ax_map.annotate(
        '  MAX +%.0fm\n  Δt +%.2fs' % (mg['gap_m'], mg['gap_t']),
        xy=((mg['sx'] + mg['fx']) / 2, (mg['sy'] + mg['fy']) / 2),
        xytext=((mg['sx'] + mg['fx']) / 2 + (
            (S['X'][view_mask].max() - S['X'][view_mask].min()) * 0.14),
                (mg['sy'] + mg['fy']) / 2 - (
            (S['Y'][view_mask].max() - S['Y'][view_mask].min()) * 0.08)),
        color=GREEN, fontsize=9, fontweight='bold',
        arrowprops=dict(arrowstyle='->', color=GREEN, lw=1.5),
        bbox=dict(boxstyle='round,pad=0.4', fc='#050f07', ec=GREEN, alpha=0.93),
        zorder=9
    )

# ── Zone boundary markers ─────────────────────────────────────────────────
for label, idx_ in [('INICI', np.argmin(np.abs(pct - Z_START))),
                     ('FINAL', np.argmin(np.abs(pct - Z_END)))]:
    xp = float(F['X'][idx_]); yp = float(F['Y'][idx_])
    ax_map.scatter(xp, yp, s=220, color=GREEN, marker='|', linewidths=2.5, zorder=9)
    if idx_ + 3 < N_PTS:
        dx_ = float(F['X'][idx_+3] - F['X'][idx_])
        dy_ = float(F['Y'][idx_+3] - F['Y'][idx_])
        n_  = np.hypot(dx_, dy_)
        px_ = (-dy_ / n_ * 20) if n_ > 0 else 0
        py_ = ( dx_ / n_ * 20) if n_ > 0 else 20
    else:
        px_, py_ = 0, 20
    ax_map.text(xp + px_, yp + py_, label,
                color=GREEN, fontsize=7.5, fontweight='bold', ha='center',
                bbox=dict(boxstyle='round,pad=0.3', fc='#050f07',
                          ec=GREEN, alpha=0.90),
                zorder=10)

# Axis extent
x_all = np.concatenate([sx_v, fx_v]); y_all = np.concatenate([sy_v, fy_v])
xr = x_all.max() - x_all.min(); yr = y_all.max() - y_all.min()
ax_map.set_xlim(x_all.min() - xr * 0.08, x_all.max() + xr * 0.08)
ax_map.set_ylim(y_all.min() - yr * 0.08, y_all.max() + yr * 0.08)
ax_map.axis('off')

# ── Mini overview inset ───────────────────────────────────────────────────
ax_mini = inset_axes(ax_map, width='32%', height='28%',
                     loc='upper right', borderpad=0.8)
ax_mini.set_facecolor('#0b0c1a')
ax_mini.plot(S['X'], S['Y'], color='#2a2a3a', lw=5, solid_capstyle='round', zorder=1)
ax_mini.plot(S['X'], S['Y'], color='#555565', lw=1.5, zorder=2)
ax_mini.plot(S['X'][zone_mask], S['Y'][zone_mask],
             color=GREEN, lw=5, solid_capstyle='round', zorder=3)
ax_mini.scatter(S['X'][zone_mask][0], S['Y'][zone_mask][0],
                s=60, color=GREEN, zorder=4, marker='o')
ax_mini.axis('off')
ax_mini.set_aspect('equal')
for sp in ax_mini.spines.values():
    sp.set_edgecolor('#303050'); sp.set_linewidth(1.2)
ax_mini.text(0.5, -0.08, 'Zona de màxim gap', transform=ax_mini.transAxes,
             color=GREEN, fontsize=7, ha='center')

# Legend
ax_map.legend(handles=[
    Line2D([0],[0], color=RED,  lw=3.5, label='01:44.845  (lenta)'),
    Line2D([0],[0], color=CYAN, lw=3.5, label='01:42.797  (ràpida)'),
    Line2D([0],[0], color=GREEN, lw=1.5, ls='--', label='gap posicional'),
    Line2D([0],[0], color=RED,  lw=0, marker='o', ms=10,
           mec='white', mew=0.8, label='ghost lent'),
    Line2D([0],[0], color=CYAN, lw=0, marker='D', ms=10,
           mec='white', mew=0.8, label='ghost ràpid'),
], loc='lower left',
    facecolor='#0c0d1a', edgecolor='#2a2a40',
    labelcolor=TXT, fontsize=9, framealpha=0.93)

# ── Header ────────────────────────────────────────────────────────────────────
fig.text(0.03, 0.955,
         'GHOST CAR  ·  ZONA DE MÀXIM GAP  ·  IMOLA GP  ·  FERRARI 296 GT3',
         color=TXT, fontsize=13, fontweight='bold', va='top')
fig.text(0.03, 0.933,
         'Zona  %.1f%% – %.1f%%   |   +%.3fs acumulats (%.0f%% del gap total)   |   '
         '01:44.845  vs  01:42.797' % (
             Z_START*100, Z_END*100, Z_GAP, Z_PCT_TOTAL),
         color=GREEN, fontsize=9.5, va='top')

# ═══════════════════════════════════════════════════════════════════════════
#  TELEMETRY CHARTS  (right side, rows 0–3)
# ═══════════════════════════════════════════════════════════════════════════

def tele_ax(row, label, slow_data, fast_data, ylabel):
    a = fig.add_subplot(gs[row, 3:])
    style_ax(a)
    a.plot(pct_v, slow_data, color=RED,  lw=2.0, alpha=0.9, label='lenta')
    a.plot(pct_v, fast_data, color=CYAN, lw=2.0, alpha=0.9, label='ràpida')
    a.fill_between(pct_v, slow_data, fast_data,
                   where=(fast_data >= slow_data), color=CYAN, alpha=0.13)
    a.fill_between(pct_v, slow_data, fast_data,
                   where=(fast_data < slow_data),  color=RED,  alpha=0.13)
    a.set_ylabel(ylabel, color=DIM, fontsize=7)
    a.set_title(label, color=TXT, fontsize=9, fontweight='bold', pad=5)
    a.legend(facecolor='#0c0d1a', edgecolor='#2a2a40',
             labelcolor=TXT, fontsize=7.5, framealpha=0.9, loc='best')
    # Zone label at top
    y_top = a.get_ylim()[1]
    a.text((Z_START + Z_END) / 2, y_top, ' MAX GAP ',
           color=GREEN, fontsize=7, fontweight='bold', ha='center', va='top',
           bbox=dict(boxstyle='round,pad=0.2', fc='#050f07', ec=GREEN, alpha=0.88))
    return a

ax_sp = tele_ax(0, 'VELOCITAT',    sspd_v, fspd_v, 'km/h')
ax_br = tele_ax(1, 'FRENADA',      sbrk_v, fbrk_v, '%')
ax_th = tele_ax(2, 'ACCELERADOR',  sthr_v, fthr_v, '%')
ax_st = tele_ax(3, 'ANGLE VOLANT', sstr_v, fstr_v, '°')
ax_st.axhline(0, color=DIM, lw=0.5, ls='--', alpha=0.5)
ax_st.set_xlabel('LapDistPct', color=DIM, fontsize=7)

# ═══════════════════════════════════════════════════════════════════════════
#  GAP RATE CHART  (bottom row, spans all right columns)
# ═══════════════════════════════════════════════════════════════════════════
ax_grate = fig.add_subplot(gs[4, 3:])
style_ax(ax_grate, show_zone=True)

# Gap accumulation (s) in view window
ax_grate.plot(pct_v, tgap_v - tgap_v[0],   # relative gain vs view start
              color=GOLD, lw=2.5, label='gap acumulat (s)')
ax_grate.fill_between(pct_v, 0, tgap_v - tgap_v[0],
                      where=((tgap_v - tgap_v[0]) > 0),
                      color=GOLD, alpha=0.22)
ax_grate.axhline(0, color=DIM, lw=0.7, ls='--', alpha=0.6)
ax_grate.set_ylabel('Δt (s)', color=DIM, fontsize=7)
ax_grate.set_xlabel('LapDistPct', color=DIM, fontsize=7)
ax_grate.set_title('GAP ACUMULAT EN AQUESTA ZONA', color=TXT,
                   fontsize=9, fontweight='bold', pad=5)

# Annotate total gain in zone
zone_gain_v = tgap[zone_mask]
zone_gain   = float(zone_gain_v[-1] - zone_gain_v[0])
ax_grate.annotate(
    '+%.3fs en la zona' % zone_gain,
    xy=(Z_END, float(np.interp(Z_END, pct_v, tgap_v - tgap_v[0]))),
    xytext=(Z_END - 0.025, float(np.interp(Z_END, pct_v, tgap_v - tgap_v[0])) + 0.04),
    color=GOLD, fontsize=8, fontweight='bold',
    arrowprops=dict(arrowstyle='->', color=GOLD, lw=1.0)
)

# ── Stats text box ────────────────────────────────────────────────────────────
avg_gap_z    = float(np.mean(tgap[zone_mask]))
max_gap_z    = float(np.max(tgap[zone_mask]))
avg_spd_diff = float(np.mean((F['Speed'][zone_mask] - S['Speed'][zone_mask]) * 3.6))
max_spd_slow = float(np.max(S['Speed'][zone_mask]) * 3.6)
max_spd_fast = float(np.max(F['Speed'][zone_mask]) * 3.6)
max_brk_slow = float(np.max(S['Brake'][zone_mask]) * 100)
max_brk_fast = float(np.max(F['Brake'][zone_mask]) * 100)
early_thr    = ''
thr_fast_idx = np.where(F['Throttle'][zone_mask] > 0.95)[0]
thr_slow_idx = np.where(S['Throttle'][zone_mask] > 0.95)[0]
if len(thr_fast_idx) and len(thr_slow_idx):
    thr_diff_pct = (pct[zone_mask][thr_slow_idx[0]] -
                    pct[zone_mask][thr_fast_idx[0]]) * TRACK_LEN
    early_thr = '\nGas ràpida: %.0fm abans' % thr_diff_pct

stats = (
    'STATS DE LA ZONA\n'
    '────────────────\n'
    'Gap acumulat: +%.3fs\n'
    'Gap max:      +%.3fs\n'
    'Δv mig:        %+.1f km/h\n'
    'Vmax lenta:   %.0f km/h\n'
    'Vmax ràpida:  %.0f km/h\n'
    'Fren. lenta:   %.0f%%\n'
    'Fren. ràpida:  %.0f%%'
    '%s'
) % (zone_gain, max_gap_z, avg_spd_diff,
     max_spd_slow, max_spd_fast,
     max_brk_slow, max_brk_fast, early_thr)

fig.text(0.975, 0.555, stats,
         color=TXT, fontsize=8, va='top', ha='right', family='monospace',
         bbox=dict(boxstyle='round,pad=0.55', fc='#0c0d1a',
                   ec='#303050', alpha=0.93))

# ── Save ─────────────────────────────────────────────────────────────────────
plt.savefig(OUT, dpi=150, bbox_inches='tight', facecolor=BG)
print("OK Zona maxim gap guardada ->", OUT)
plt.close()


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


OK Zona maxim gap: 45.1% - 50.1%  -> +0.631s  (30% del gap total)
OK Zona maxim gap guardada -> c:\Users\AvAd738\OneDrive - HP Inc\Documents\Adriana\AI-coaching-for-iRacing\ghost_maxgap_visualization.png
